In [3]:
import polars as pl
import matplotlib.pyplot as plt
import numpy as np
import os
import sklearn
import torch
from torchvision import transforms
from google.colab import ai

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
csv_path = "/content/drive/MyDrive/STINTSYMCO1/FracAtlas/dataset.csv"
df = pl.read_csv(csv_path)
df.head()


image_id,hand,leg,hip,shoulder,mixed,hardware,multiscan,fractured,fracture_count,frontal,lateral,oblique
str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""IMG0000000.jpg""",0,1,0,0,0,0,1,0,0,1,1,0
"""IMG0000001.jpg""",0,1,0,0,0,0,1,0,0,1,1,0
"""IMG0000002.jpg""",0,1,0,0,0,0,1,0,0,1,1,0
"""IMG0000003.jpg""",0,1,0,0,0,0,1,0,0,0,1,1
"""IMG0000004.jpg""",0,1,0,0,0,0,1,0,0,0,1,1


In [10]:
fractured = df.filter(df["fractured"] == 1)
nonfractured = df.filter(df["fractured"] == 0)

train, test_validate = sklearn.model_selection.train_test_split(df, stratify=df["fractured"], train_size=0.7, test_size=0.3, random_state=42)
test, validation = sklearn.model_selection.train_test_split(test_validate, stratify=test_validate["fractured"], train_size=0.5, test_size=0.5, random_state=42)


train_paths = train.with_columns(
    pl.when(pl.col("fractured") == 1).then("../FracAtlas/images/Fractured/" + pl.col("image_id")).otherwise("../FracAtlas/images/Non_fractured/" + pl.col("image_id")).alias("paths")
)

train_paths = train_paths["paths"]
train_labels = train["fractured"]


In [11]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),                          
    transforms.Normalize([0.485, 0.456, 0.406],    
                         [0.229, 0.224, 0.225])   
])
